# 02 — Pré-processamento

Cada decisão é documentada. Transformações que aprendem parâmetros — imputação, escala e One-Hot Encoding — permanecem dentro das pipelines do notebook 03 para evitar vazamento.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import RANDOM_STATE, FIGURES, METRICS, MODELS, TARGET

FIGURES.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
print("RANDOM_STATE =", RANDOM_STATE)

from src.data import load_raw, save_processed
from src.preprocessing import check_missing, build_target, build_processed_dataset

RANDOM_STATE = 42


In [2]:
application, credit = load_raw()

## 1. Dados faltantes

In [3]:
display(check_missing(application))

,nulos,pct
OCCUPATION_TYPE,134203,30.6


**Decisão.** `OCCUPATION_TYPE` recebe a categoria explícita `Unknown`. O código `DAYS_EMPLOYED = 365243` é convertido em ausência na variável `EMPLOYMENT_YEARS` e registrado em uma flag. A imputação de `EMPLOYMENT_YEARS` será feita pela mediana **dentro da pipeline**.

## 2. Definição da variável alvo

- `TARGET_BAD = 1`: pelo menos uma ocorrência de `STATUS` entre 1 e 5, equivalente a atraso de **30 dias ou mais**;
- `TARGET_BAD = 0`: histórico observável sem `STATUS` entre 1 e 5;
- históricos exclusivamente `X` são excluídos.

In [4]:
application_unique, target_cliente = build_target(application, credit)
assert len(set(application_unique["ID"]).intersection(set(credit["ID"]))) == 36457
assert len(target_cliente) == 33110
assert int((target_cliente==0).sum()) == 28819
assert int((target_cliente==1).sum()) == 4291
display(pd.DataFrame({"quantidade":target_cliente.value_counts().sort_index(),"percentual":(target_cliente.value_counts(normalize=True).sort_index()*100).round(2)}))

,quantidade,percentual
TARGET_BAD,,
0,28819,87.04
1,4291,12.96


**Justificativa.** A regra de 30+ dias preserva uma classe positiva com **4.291 clientes (12,96%)**, quantidade suficiente para modelagem. Limiares mais severos reduziriam a classe positiva drasticamente. A mesma regra é mantida em todo o projeto.

## 3. Normalização / padronização

**Escolha e justificativa.** O arquivo tratado não é escalado previamente. KNN recebe `StandardScaler` dentro da pipeline; modelos baseados em árvores não necessitam padronização. Isso impede que parâmetros de escala sejam calculados usando o teste.

## 4. Feature engineering

São criadas `AGE_YEARS`, `EMPLOYMENT_YEARS`, `FLAG_EMPLOYMENT_SENTINEL` e `FLAG_FAMILY_INCONSISTENT`; indicadores Y/N de patrimônio são convertidos para 0/1. As colunas originais substituídas são removidas.

In [5]:
base = build_processed_dataset(application, credit)
assert base.shape == (33110,20)
assert base["ID"].nunique() == 33110
assert base[TARGET].isna().sum() == 0
assert base["OCCUPATION_TYPE"].isna().sum() == 0
assert base["FLAG_EMPLOYMENT_SENTINEL"].sum() == 5642
assert base["EMPLOYMENT_YEARS"].isna().sum() == 5642
assert base["FLAG_FAMILY_INCONSISTENT"].sum() == 2
display(base.head())

,ID,AMT_INCOME_TOTAL,AGE_YEARS,EMPLOYMENT_YEARS,CNT_CHILDREN,CNT_FAM_MEMBERS,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,FLAG_EMPLOYMENT_SENTINEL,FLAG_FAMILY_INCONSISTENT,CODE_GENDER,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,TARGET_BAD
0,5008804,427500.0,32.867899,12.435318,0,2,1,1,1,0,0,0,0,M,Working,Higher education,Civil marriage,Rented apartment,Unknown,1
1,5008805,427500.0,32.867899,12.435318,0,2,1,1,1,0,0,0,0,M,Working,Higher education,Civil marriage,Rented apartment,Unknown,1
2,5008806,112500.0,58.792608,3.104723,0,2,1,1,0,0,0,0,0,M,Working,Secondary / secondary special,Married,House / apartment,Security staff,0
3,5008808,270000.0,52.320329,8.353183,0,1,0,1,0,1,1,0,0,F,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,Sales staff,0
5,5008810,270000.0,52.320329,8.353183,0,1,0,1,0,1,1,0,0,F,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,Sales staff,0


**Interpretação.** As transformações preservam toda a população elegível. O código especial de emprego deixa de ser interpretado como tempo literal, e sua ocorrência continua rastreável pela flag. Não são criadas features especulativas sem fundamento no dado disponível.

## 5. Salvar dataset tratado

In [6]:
save_processed(base)
print("Arquivo salvo em data/processed/dataset_tratado.csv")
print("Dimensão:", base.shape)

Arquivo salvo em data/processed/dataset_tratado.csv
Dimensão: (33110, 20)


**Resultado.** A base tratada contém **33.110 clientes**, 18 features, `ID` e `TARGET_BAD`. O arquivo em `data/processed/` é regenerável e, conforme o template, não deve ser versionado no GitHub.